# EM synapse mapping - multi-neuron use case

Creating a circuit from a set of morphologies with spines.

# Authentication

Follow the instructions below to authenticate and select a project to work in.

The morphology-with-spines instances that are to serve as the input of circuit generation must be accessible from the project.

In [ ]:
import obi_auth

from obi_notebook.get_projects import get_projects
from obi_notebook.get_entities import get_entities

from entitysdk.models import CellMorphology, EMCellMesh, EMDenseReconstructionDataset, Circuit
from entitysdk import Client
from entitysdk._server_schemas import AssetLabel

token = obi_auth.get_token(environment="staging", auth_mode="daf")
project_context = get_projects(token, env="staging")

# Find skeletonized morphologies as input

We search for morphologies originating from the `IARPA MICrONS mouse`, as they are derived by skeletonization

In [ ]:
entity_client = Client(token_manager=token, project_context=project_context, environment="staging")

results = entity_client.search_entity(
    entity_type=CellMorphology,
    query={"subject__name": "IARPA MICrONS mouse"},
).all()

# Filter by morphology name, avoid duplicates
morph_names = []
spiny_morphologies = []
for morph in results:
    if morph.name not in morph_names:
        morph_names.append(morph.name)
        spiny_morphologies.append(morph)

print(f"Found {len(spiny_morphologies)} spiny morphologies:")
for m in spiny_morphologies:
    print(f"  - {m.name} (id={m.id})")


Select > 1 entity from the above (copy and past below)

In [ ]:
selected_entity_ids = [
    "40f3aa77-48ba-42e0-bc82-764ef3deee72",
    "bcc60e2e-5daf-427e-b008-3bc8ca16289c"
]

In [ ]:
import caveclient
temporary_client = caveclient.CAVEclient()
temporary_client.auth.get_new_token()

Paste your token below:

In [ ]:
import os
import caveclient
os.environ["CAVECLIENT_MICRONS_API_KEY"] = "INSERT TOKEN HERE!"

In [ ]:
from obi_one.scientific.from_id.cell_morphology_from_id import CellMorphologyFromID
circuit_neurons = [CellMorphologyFromID(id_str=id_) for id_ in selected_entity_ids]

# Set up the multi-neuron synapse mapping task

In [ ]:
from obi_one.scientific.tasks.em_synapse_mapping.config import EMSynapseMappingSingleConfig, EMSynapseMappingInputNamedTuple, AdvancedEMSynapseMappingOptions
from obi_one.scientific.tasks.em_synapse_mapping.task import EMSynapseMappingTask
from obi_one.scientific.from_id.named_tuple_from_id import EMSynapseMappingInputNamedTuple
from obi_one.core.info import Info
from pathlib import Path

# We take the last 3 characters of the morphology UUID to easily identify the neurons
circuit_neurons_short_names = [n.id_str[-3:] for n in circuit_neurons]

OUTPUT_DIR = "multi-neuron-circuit-" + "-".join(circuit_neurons_short_names)
INCLUDE_SPINY_MORPH = True
suffix = 1
while Path(OUTPUT_DIR).exists():
    OUTPUT_DIR = "multi-neuron-circuit-" + "-".join(circuit_neurons_short_names) + f"_{suffix}"
    suffix += 1

task_config = EMSynapseMappingSingleConfig(
    info=Info(
        campaign_name="EM Synapse Mapping Multi-Neuron",
        campaign_description="Map EM synapses onto multiple spiny morphologies"
    ),
    initialize=EMSynapseMappingSingleConfig.Initialize(
        neurons=EMSynapseMappingInputNamedTuple(name="Skeletons", elements=tuple(circuit_neurons)),
    ),
    coordinate_output_root=OUTPUT_DIR,
    advanced_options=AdvancedEMSynapseMappingOptions(include_spiny_morphologies=INCLUDE_SPINY_MORPH)
)

task = EMSynapseMappingTask(config=task_config)
print(f"Running synapse mapping task with {len(circuit_neurons)} neurons...")
print(f"Output: {OUTPUT_DIR}/")
task.execute(db_client=entity_client)

## Inspect the output (local files)

In [ ]:
import json
import h5py
from pathlib import Path

output_dir = Path(OUTPUT_DIR)
config_path = output_dir / "SONATA/circuit_config.json"

if config_path.exists():
    with open(config_path) as f:
        circuit_cfg = json.load(f)
    print(json.dumps(circuit_cfg, indent=2))

    edges_file = output_dir / "synaptome-edges.h5"
    if edges_file.exists():
        with h5py.File(edges_file, "r") as h5:
            print("\nEdge populations:")
            for pop_name in h5["edges"]:
                grp = h5[f"edges/{pop_name}"]
                n = len(grp["source_node_id"])
                src_pop = grp["source_node_id"].attrs.get("node_population", "?")
                tgt_pop = grp["target_node_id"].attrs.get("node_population", "?")
                print(f"  {pop_name}: {n} edges ({src_pop} -> {tgt_pop})")
else:
    print("No circuit output found. Run the task cell above first.")

## Test circuit validity with SNAP

Display the biophysical node population properties (one row per neuron)


In [ ]:
import bluepysnap as snap

circ = snap.Circuit(OUTPUT_DIR + "/SONATA/circuit_config.json")
bio_pop = circ.nodes["biophysical_neurons"]
bio_pop.get()

Verify that SWC and HDF5 morphology files referenced by the circuit can be loaded

In [ ]:
print(f"SWC: {bio_pop.morph.get(0, extension="swc")}")
print(f"HDF5: {bio_pop.morph.get(0, extension="h5")}")

Test that they are translated to the origin.

In [ ]:
from neurom.core import Morphology
from neurom.view import plot_morph

from matplotlib import pyplot as plt

m_untrans = [
    bio_pop.morph.get(i, extension="swc", transform=False)
    for i in range(bio_pop.count())
]
m_trans = [
    bio_pop.morph.get(i, extension="swc", transform=True)
    for i in range(bio_pop.count())
]

for i, m in enumerate(m_untrans):
    if i == 0:
        plot_morph(Morphology(m))
    else:
        plot_morph(Morphology(m), ax=plt.gca())
for i, m in enumerate(m_trans):
    if i == 0:
        plot_morph(Morphology(m))
    else:
        plot_morph(Morphology(m), ax=plt.gca())

Load the first morphology of the circuit as a spiny morphology

In [ ]:
from morph_spines import load_morphology_with_spines

if INCLUDE_SPINY_MORPH:
    spiny_morph = load_morphology_with_spines(bio_pop.config["alternate_morphologies"]["h5v1"],
                                bio_pop.get(0, properties="morphology").split("/")[1])

    plot_morph(spiny_morph.morphology)